# Binary ROI Classifier Evaluation

Evaluates the trained eye and mouth classifiers across multiple datasets and reports accuracy, precision/recall/F1, confusion matrix, and ROC AUC.

**Datasets:**
- `own_eyes` / `own_mouth` - held-out test split from `dataset_split/`
- `public_eyes` - external eye dataset (`public_eyes_dataset/test`)
- `public_mouth` - external mouth dataset (`public_mouth_dataset`, no split, evaluated whole)

## Imports

In [1]:
from __future__ import annotations

import sys
import os
from pathlib import Path

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from torch.utils.data import DataLoader
from tqdm import tqdm

project_root = Path(os.getcwd()).parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from model_architecture.dataset.binary_classifier_dataset import BinaryClassifierDataset, _VAL_TRANSFORMS
from model_architecture.models.binary_roi_classifier import build_binary_classifier

## Config

In [2]:
CHECKPOINT_DIR = Path("runs/binary")
RESULTS_DIR = Path("runs/eval")

# Checkpoint filename per ROI
CHECKPOINTS = {
    "eyes": "best_eyes.pt",
    "mouth": "best_mouth.pt",
}

CLASS_LABELS = {
    "eyes": {"closed": 0, "open": 1},
    "mouth": {"closed": 0, "open": 1},
}

# Datasets to evaluate
DATASETS = {
    "own_eyes": ("eyes", "../dataset_split/test/eyes"),
    "own_mouth": ("mouth", "../dataset_split/test/mouth"),
    "public_eyes": ("eyes", "../public_eyes_dataset/test"),
    "public_mouth": ("mouth", "../public_mouth_dataset"),
}

BATCH_SIZE = 64
NUM_WORKERS = 4

## Data & Inference Helpers

In [3]:
def load_test_loader(roi: str, root: Path) -> DataLoader:
    dataset = BinaryClassifierDataset(root, CLASS_LABELS[roi], transform=_VAL_TRANSFORMS)
    print(f"Test samples: {len(dataset)}")

    label_counts = {}
    for _, label in dataset.samples:
        label_counts[label] = label_counts.get(label, 0) + 1
    for label, count in sorted(label_counts.items()):
        class_name = [k for k, v in CLASS_LABELS[roi].items() if v == label][0]
        print(f"{class_name}: {count}")

    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )


def run_inference(model, loader, device) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Returns (all_labels, all_preds, all_probs)."""
    model.eval()
    all_labels, all_preds, all_probs = [], [], []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Evaluating"):
            images = images.to(device)
            logits = model(images)
            probs = torch.softmax(logits, dim=1)

            all_preds.extend(logits.argmax(dim=1).cpu().tolist())
            all_probs.extend(probs[:, 1].cpu().tolist())
            all_labels.extend(labels.tolist())

    return (
        np.array(all_labels),
        np.array(all_preds),
        np.array(all_probs),
    )

## Plotting

In [4]:
def plot_confusion_matrix(
    cm: np.ndarray,
    class_names: list[str],
    title_suffix: str,
    save_path: Path,
) -> None:
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names,
        ax=ax,
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title(f"Confusion Matrix - {title_suffix}")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"Saved confusion matrix -> {save_path}")


def plot_roc_curve(
    labels: np.ndarray,
    probs: np.ndarray,
    title_suffix: str,
    save_path: Path,
) -> None:
    fpr, tpr, _ = roc_curve(labels, probs)
    auc = roc_auc_score(labels, probs)

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(fpr, tpr, label=f"AUC = {auc:.3f}", color="steelblue", lw=2)
    ax.plot([0, 1], [0, 1], "k--", lw=1)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(f"ROC Curve - {title_suffix}")
    ax.legend(loc="lower right")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"Saved ROC curve -> {save_path}")

## Evaluation

In [5]:
_MODEL_CACHE: dict[str, nn.Module] = {}


def get_model(roi: str, device: torch.device) -> nn.Module | None:
    if roi in _MODEL_CACHE:
        return _MODEL_CACHE[roi]

    checkpoint_path = CHECKPOINT_DIR / roi / CHECKPOINTS[roi]
    if not checkpoint_path.exists():
        print(f"ERROR: No checkpoint found at {checkpoint_path}")
        return None

    model = build_binary_classifier(pretrained=False, freeze_backbone=False).to(device)
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    print(f"Loaded checkpoint: {checkpoint_path}")
    _MODEL_CACHE[roi] = model
    return model


def evaluate_dataset(dataset_name: str, roi: str, root: str, device: torch.device) -> dict | None:
    print(f"\n{'='*60}")
    print(f"Evaluating: {dataset_name} (ROI={roi.upper()}) on {device}")
    print(f"{'='*60}")

    root_path = Path(root)
    if not root_path.exists():
        print(f"ERROR: Dataset root not found: {root_path}")
        return None

    model = get_model(roi, device)
    if model is None:
        return None

    results_dir = RESULTS_DIR / dataset_name
    results_dir.mkdir(parents=True, exist_ok=True)

    print("\nTest set:")
    loader = load_test_loader(roi, root_path)

    labels, preds, probs = run_inference(model, loader, device)

    class_names = [k for k, v in sorted(CLASS_LABELS[roi].items(), key=lambda x: x[1])]
    acc = accuracy_score(labels, preds)
    precision_macro = precision_score(labels, preds, average="macro", zero_division=0)
    recall_macro = recall_score(labels, preds, average="macro", zero_division=0)
    f1_macro = f1_score(labels, preds, average="macro", zero_division=0)
    precision_pos = precision_score(labels, preds, pos_label=1, zero_division=0)
    recall_pos = recall_score(labels, preds, pos_label=1, zero_division=0)
    f1_pos = f1_score(labels, preds, pos_label=1, zero_division=0)
    try:
        auc = roc_auc_score(labels, probs)
    except ValueError:
        auc = float("nan")
    cm = confusion_matrix(labels, preds, labels=list(range(len(class_names))))

    print(f"\n--- Results ---")
    print(f"Accuracy: {acc:.4f}")
    print(f"ROC AUC: {auc:.4f}")
    print(f"Precision (macro): {precision_macro:.4f}  |  Precision (open): {precision_pos:.4f}")
    print(f"Recall    (macro): {recall_macro:.4f}  |  Recall    (open): {recall_pos:.4f}")
    print(f"F1        (macro): {f1_macro:.4f}  |  F1        (open): {f1_pos:.4f}")
    print(f"\nClassification Report:")
    print(classification_report(labels, preds, target_names=class_names, zero_division=0))
    print(f"Confusion Matrix:")
    print(f"Labels: {class_names}")
    print(f"   {cm}")

    checkpoint_path = CHECKPOINT_DIR / roi / CHECKPOINTS[roi]
    report_path = results_dir / "test_report.txt"
    with open(report_path, "w") as f:
        f.write(f"Dataset: {dataset_name}\n")
        f.write(f"ROI: {roi.upper()}\n")
        f.write(f"Root: {root_path}\n")
        f.write(f"Checkpoint: {checkpoint_path}\n\n")
        f.write(f"Accuracy: {acc:.4f}\n")
        f.write(f"ROC AUC: {auc:.4f}\n")
        f.write(f"Precision (macro): {precision_macro:.4f}    Precision (open): {precision_pos:.4f}\n")
        f.write(f"Recall    (macro): {recall_macro:.4f}    Recall    (open): {recall_pos:.4f}\n")
        f.write(f"F1        (macro): {f1_macro:.4f}    F1        (open): {f1_pos:.4f}\n\n")
        f.write("Classification Report:\n")
        f.write(classification_report(labels, preds, target_names=class_names, zero_division=0))
        f.write(f"\nConfusion Matrix ({class_names}):\n")
        f.write(str(cm))
    print(f"\nSaved report -> {report_path}")

    title = f"{dataset_name} [{roi.upper()}]"
    plot_confusion_matrix(cm, class_names, title, results_dir / "test_confusion_matrix.png")
    plot_roc_curve(labels, probs, title, results_dir / "test_roc_curve.png")

    return {
        "dataset": dataset_name,
        "roi": roi,
        "n": int(len(labels)),
        "accuracy": acc,
        "roc_auc": auc,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "precision_open": precision_pos,
        "recall_open": recall_pos,
        "f1_open": f1_pos,
    }

## Setup Run

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
summary = []

### Own Eyes - "dataset_split/test/eyes"

In [7]:
name = "own_eyes"
roi, root = DATASETS[name]
result = evaluate_dataset(name, roi, root, device)
if result is not None:
    summary.append(result)


Evaluating: own_eyes (ROI=EYES) on cuda
Loaded checkpoint: runs\binary\eyes\best_eyes.pt

Test set:
Test samples: 4118
closed: 405
open: 3713


Evaluating: 100%|██████████| 65/65 [00:28<00:00,  2.27it/s]


--- Results ---
Accuracy: 0.9769
ROC AUC: 0.9904
Precision (macro): 0.9469  |  Precision (open): 0.9837
Recall    (macro): 0.9201  |  Recall    (open): 0.9908
F1        (macro): 0.9330  |  F1        (open): 0.9873

Classification Report:
              precision    recall  f1-score   support

      closed       0.91      0.85      0.88       405
        open       0.98      0.99      0.99      3713

    accuracy                           0.98      4118
   macro avg       0.95      0.92      0.93      4118
weighted avg       0.98      0.98      0.98      4118

Confusion Matrix:
Labels: ['closed', 'open']
   [[ 344   61]
 [  34 3679]]

Saved report -> runs\eval\own_eyes\test_report.txt
Saved confusion matrix -> runs\eval\own_eyes\test_confusion_matrix.png
Saved ROC curve -> runs\eval\own_eyes\test_roc_curve.png


### Own Mouth - "dataset_split/test/mouth"

In [8]:
name = "own_mouth"
roi, root = DATASETS[name]
result = evaluate_dataset(name, roi, root, device)
if result is not None:
    summary.append(result)


Evaluating: own_mouth (ROI=MOUTH) on cuda
Loaded checkpoint: runs\binary\mouth\best_mouth.pt

Test set:
Test samples: 2085
closed: 2048
open: 37


Evaluating: 100%|██████████| 33/33 [00:27<00:00,  1.19it/s]


--- Results ---
Accuracy: 0.9923
ROC AUC: 0.9993
Precision (macro): 0.8491  |  Precision (open): 0.6981
Recall    (macro): 0.9961  |  Recall    (open): 1.0000
F1        (macro): 0.9092  |  F1        (open): 0.8222

Classification Report:
              precision    recall  f1-score   support

      closed       1.00      0.99      1.00      2048
        open       0.70      1.00      0.82        37

    accuracy                           0.99      2085
   macro avg       0.85      1.00      0.91      2085
weighted avg       0.99      0.99      0.99      2085

Confusion Matrix:
Labels: ['closed', 'open']
   [[2032   16]
 [   0   37]]

Saved report -> runs\eval\own_mouth\test_report.txt
Saved confusion matrix -> runs\eval\own_mouth\test_confusion_matrix.png
Saved ROC curve -> runs\eval\own_mouth\test_roc_curve.png


### Public Eyes - "public_eyes_dataset/test"

In [9]:
name = "public_eyes"
roi, root = DATASETS[name]
result = evaluate_dataset(name, roi, root, device)
if result is not None:
    summary.append(result)


Evaluating: public_eyes (ROI=EYES) on cuda

Test set:
Test samples: 6991
closed: 1666
open: 5325


Evaluating: 100%|██████████| 110/110 [00:28<00:00,  3.79it/s]


--- Results ---
Accuracy: 0.8863
ROC AUC: 0.9174
Precision (macro): 0.8429  |  Precision (open): 0.9264
Recall    (macro): 0.8447  |  Recall    (open): 0.9241
F1        (macro): 0.8438  |  F1        (open): 0.9253

Classification Report:
              precision    recall  f1-score   support

      closed       0.76      0.77      0.76      1666
        open       0.93      0.92      0.93      5325

    accuracy                           0.89      6991
   macro avg       0.84      0.84      0.84      6991
weighted avg       0.89      0.89      0.89      6991

Confusion Matrix:
Labels: ['closed', 'open']
   [[1275  391]
 [ 404 4921]]

Saved report -> runs\eval\public_eyes\test_report.txt
Saved confusion matrix -> runs\eval\public_eyes\test_confusion_matrix.png
Saved ROC curve -> runs\eval\public_eyes\test_roc_curve.png


### Public Mouth - "public_mouth_dataset"

In [10]:
name = "public_mouth"
roi, root = DATASETS[name]
result = evaluate_dataset(name, roi, root, device)
if result is not None:
    summary.append(result)


Evaluating: public_mouth (ROI=MOUTH) on cuda

Test set:
Test samples: 5119
closed: 2591
open: 2528


Evaluating: 100%|██████████| 80/80 [00:28<00:00,  2.83it/s]


--- Results ---
Accuracy: 0.8242
ROC AUC: 0.9112
Precision (macro): 0.8360  |  Precision (open): 0.8959
Recall    (macro): 0.8230  |  Recall    (open): 0.7286
F1        (macro): 0.8222  |  F1        (open): 0.8037

Classification Report:
              precision    recall  f1-score   support

      closed       0.78      0.92      0.84      2591
        open       0.90      0.73      0.80      2528

    accuracy                           0.82      5119
   macro avg       0.84      0.82      0.82      5119
weighted avg       0.84      0.82      0.82      5119

Confusion Matrix:
Labels: ['closed', 'open']
   [[2377  214]
 [ 686 1842]]

Saved report -> runs\eval\public_mouth\test_report.txt
Saved confusion matrix -> runs\eval\public_mouth\test_confusion_matrix.png
Saved ROC curve -> runs\eval\public_mouth\test_roc_curve.png


## Summary

In [11]:
print(f"{'='*88}")
print("Summary (macro-averaged across classes)")
print(f"{'='*88}")
header = f"{'dataset':<16} {'roi':<6} {'n':>6} {'acc':>7} {'auc':>7} {'prec':>7} {'rec':>7} {'f1':>7}"
print(header)
print("-" * len(header))
for r in summary:
    print(
        f"{r['dataset']:<16} {r['roi']:<6} {r['n']:>6} "
        f"{r['accuracy']:>7.4f} {r['roc_auc']:>7.4f} "
        f"{r['precision_macro']:>7.4f} {r['recall_macro']:>7.4f} {r['f1_macro']:>7.4f}"
    )

print(f"\n{'='*88}")
print("Summary (positive class = 'open')")
print(f"{'='*88}")
header = f"{'dataset':<16} {'roi':<6} {'n':>6} {'prec_open':>10} {'rec_open':>10} {'f1_open':>10}"
print(header)
print("-" * len(header))
for r in summary:
    print(
        f"{r['dataset']:<16} {r['roi']:<6} {r['n']:>6} "
        f"{r['precision_open']:>10.4f} {r['recall_open']:>10.4f} {r['f1_open']:>10.4f}"
    )

Summary (macro-averaged across classes)
dataset          roi         n     acc     auc    prec     rec      f1
----------------------------------------------------------------------
own_eyes         eyes     4118  0.9769  0.9904  0.9469  0.9201  0.9330
own_mouth        mouth    2085  0.9923  0.9993  0.8491  0.9961  0.9092
public_eyes      eyes     6991  0.8863  0.9174  0.8429  0.8447  0.8438
public_mouth     mouth    5119  0.8242  0.9112  0.8360  0.8230  0.8222

Summary (positive class = 'open')
dataset          roi         n  prec_open   rec_open    f1_open
---------------------------------------------------------------
own_eyes         eyes     4118     0.9837     0.9908     0.9873
own_mouth        mouth    2085     0.6981     1.0000     0.8222
public_eyes      eyes     6991     0.9264     0.9241     0.9253
public_mouth     mouth    5119     0.8959     0.7286     0.8037
